# AHR v2 — exact Go rules and replay

This notebook supplies the fresh Go-state contract, exact legal transitions, history-sensitive replay and an independently implemented transition/scorer. It imports no old models, training or datasets.

**Main experiment:** 9×9, komi 7.5, area scoring, no suicide, positional superko, passes exempt from superko, and two consecutive passes end the game. There is **no automatic dead-stone adjudication**: stones must be captured before passing. An early-pass result can therefore differ from an engine that removes presumed dead stones.

**Inputs:** Python standard library for the engine; deterministic fixtures in this notebook; project Jupyter/PyTorch dependencies and only the tagged definitions from notebook 00 for the source-bound receipt. **Outputs:** tagged `ahr-export` definitions, a bounded validation receipt, and saved check results. **Expected runtime:** under two minutes on CPU. **Checkpoint:** none; there is no training or persistent generation. **Exact resume:** open a fresh project kernel and Run All from the first cell. Every fixture is deterministic and the receipt is replaced atomically only after all checks pass.

Black is `+1`, White is `−1`, and empty is `0`. Actions are flat row-major points; action `size²` is pass. Outcomes explicitly distinguish Black's perspective from the current player to move. A nonterminal board score is a static area count, not a proved game result.

## 1. Freeze the state contract

A state carries the entire positional-superko set, eight recent board frames, the complete action chronology from its setup root, and the setup root itself. Fixtures may use smaller boards; the learning and arena contract remains 9×9. Imported definitions come only from the tagged cells below.

In [ ]:
from dataclasses import dataclass
from functools import lru_cache
from math import isfinite
from typing import Mapping
import hashlib
import json

GO2_RULE_PROFILE = 'area_no_suicide_psk_v1'
GO2_RULES = {
    'profile': GO2_RULE_PROFILE, 'main_size': 9, 'main_komi': 7.5,
    'scoring': 'area', 'suicide': False, 'ko': 'positional_superko',
    'pass_exempt_superko': True, 'terminal_passes': 2,
    'dead_stones': 'must_be_captured_before_passing',
    'recent_board_frames': 8, 'move_cap': None,
}
GO2_BLACK, GO2_WHITE, GO2_EMPTY, GO2_PASS = 1, -1, 0, 81

class Go2IllegalMove(ValueError):
    """Requested action is illegal under the frozen rule profile."""

def go2_validate_board(board, size):
    values = tuple(board)
    if len(values) != size * size or any(type(x) is not int or x not in (-1, 0, 1) for x in values):
        raise ValueError('Board must contain size² integer stones from {-1, 0, +1}.')
    return values

@dataclass(frozen=True, slots=True)
class Go2State:
    board: tuple[int, ...]
    to_play: int = 1
    size: int = 9
    komi: float = 7.5
    rule_profile: str = GO2_RULE_PROFILE
    recent_boards: tuple[tuple[int, ...], ...] = ()
    positional_history: frozenset[tuple[int, ...]] = frozenset()
    consecutive_passes: int = 0
    actions: tuple[int, ...] = ()
    initial_board: tuple[int, ...] = ()
    initial_to_play: int = 1
    initial_positional_history: frozenset[tuple[int, ...]] = frozenset()

    def __post_init__(self):
        if type(self.size) is not int or self.size < 2:
            raise ValueError('size must be an integer of at least 2.')
        if type(self.to_play) is not int or self.to_play not in (-1, 1):
            raise ValueError('to_play must be integer +1 or -1.')
        if type(self.initial_to_play) is not int or self.initial_to_play not in (-1, 1):
            raise ValueError('initial_to_play must be integer +1 or -1.')
        if type(self.komi) not in (int, float) or not isfinite(self.komi):
            raise ValueError('komi must be a finite real number.')
        if self.rule_profile != GO2_RULE_PROFILE:
            raise ValueError('Unsupported rule profile; do not silently change scoring or ko rules.')
        board = go2_validate_board(self.board, self.size)
        initial = go2_validate_board(self.initial_board or board, self.size)
        frames = tuple(go2_validate_board(x, self.size) for x in (self.recent_boards or (board,)))
        if not 1 <= len(frames) <= GO2_RULES['recent_board_frames'] or frames[-1] != board:
            raise ValueError('Recent boards must contain 1–8 chronological frames ending at current board.')
        # Validate before deduplication: bools/floats can compare equal to integer stones.
        history = frozenset(go2_validate_board(x, self.size) for x in self.positional_history) | {board}
        initial_history = frozenset(go2_validate_board(x, self.size) for x in self.initial_positional_history) | {initial}
        if not initial_history <= history or any(frame not in history for frame in frames):
            raise ValueError('Initial and recent boards must be retained in positional history.')
        actions = tuple(self.actions)
        if any(type(x) is not int or not 0 <= x <= self.size * self.size for x in actions):
            raise ValueError('Chronology contains an invalid action.')
        expected_player = self.initial_to_play * (-1 if len(actions) % 2 else 1)
        if self.to_play != expected_player:
            raise ValueError('Current player disagrees with action chronology and setup player.')
        trailing_passes = 0
        for action in reversed(actions):
            if action != self.size * self.size: break
            trailing_passes += 1
        if type(self.consecutive_passes) is not int or self.consecutive_passes not in (0, 1, 2):
            raise ValueError('Consecutive passes must be integer 0, 1 or 2.')
        if trailing_passes != self.consecutive_passes:
            raise ValueError('Pass counter disagrees with complete action chronology.')
        for name, value in (('board', board), ('initial_board', initial), ('recent_boards', frames),
                            ('positional_history', history), ('initial_positional_history', initial_history),
                            ('actions', actions), ('komi', float(self.komi))):
            object.__setattr__(self, name, value)

    @property
    def terminal(self): return self.consecutive_passes == 2
    @property
    def pass_action(self): return self.size * self.size
    @property
    def move_number(self): return len(self.actions)
    @property
    def history(self): return self.positional_history
    def to_dict(self): return go2_state_to_dict(self)
    @classmethod
    def from_dict(cls, value): return go2_state_from_dict(value)


def go2_initial_state(size=9, komi=7.5):
    return go2_state_from_board((0,) * (size * size), size=size, komi=komi)

def go2_state_from_board(board, *, to_play=1, size=9, komi=7.5, positional_history=()):
    """Explicit setup root; supplied boards are shape-validated, not claimed reachable from empty."""
    values = tuple(board)
    history = tuple(tuple(x) for x in positional_history) + (values,)
    return Go2State(board=values, to_play=to_play, size=size, komi=komi,
                    positional_history=history, initial_board=values,
                    initial_to_play=to_play, initial_positional_history=history)


def go2_state_to_dict(state):
    return {
        'schema': 'go2_state_v1', 'size': state.size, 'board': list(state.board),
        'to_play': state.to_play, 'komi': state.komi, 'rule_profile': state.rule_profile,
        'recent_boards': [list(x) for x in state.recent_boards],
        'positional_history': [list(x) for x in sorted(state.positional_history)],
        'consecutive_passes': state.consecutive_passes, 'actions': list(state.actions),
        'initial_board': list(state.initial_board), 'initial_to_play': state.initial_to_play,
        'initial_positional_history': [list(x) for x in sorted(state.initial_positional_history)],
    }


def go2_state_from_dict(value: Mapping):
    if value.get('schema') != 'go2_state_v1':
        raise ValueError('State schema is missing or unsupported.')
    required = {'size','board','to_play','komi','rule_profile','recent_boards',
                'positional_history','consecutive_passes','actions','initial_board',
                'initial_to_play','initial_positional_history'}
    if set(value) != required | {'schema'}:
        raise ValueError('State must retain the exact complete state schema.')
    return Go2State(**{key: value[key] for key in required})


## 2. Exact legal transitions

A placement captures adjacent opponent groups before testing suicide, then rejects any previously seen board. Pass leaves the board unchanged and is explicitly exempt from superko. Terminal states have no legal action. The fast successor preserves validated immutable histories without rescanning every previous board at each search node.

In [ ]:
@lru_cache(maxsize=16)
def go2_neighbors(size):
    return tuple(tuple(rr * size + cc for rr, cc in ((r-1,c),(r+1,c),(r,c-1),(r,c+1))
                       if 0 <= rr < size and 0 <= cc < size)
                 for r in range(size) for c in range(size))


def go2_group(board, size, point):
    values = tuple(board)
    if type(point) is not int or not 0 <= point < size * size or values[point] == 0:
        raise ValueError('point must contain a stone.')
    color = values[point]
    stones, liberties, pending = {point}, set(), [point]
    adjacent = go2_neighbors(size)
    while pending:
        for neighbor in adjacent[pending.pop()]:
            if values[neighbor] == 0: liberties.add(neighbor)
            elif values[neighbor] == color and neighbor not in stones:
                stones.add(neighbor); pending.append(neighbor)
    return frozenset(stones), frozenset(liberties)


def go2_groups(board, size):
    seen, result = set(), []
    for point, color in enumerate(board):
        if color and point not in seen:
            stones, liberties = go2_group(board, size, point)
            seen.update(stones); result.append((stones, liberties))
    return tuple(result)


def go2_successor(state, board, passes, action):
    """Internal trusted successor: only go2_step calls this after exact rule checks."""
    child = object.__new__(Go2State)
    fields = {
        'board': board, 'to_play': -state.to_play, 'size': state.size,
        'komi': state.komi, 'rule_profile': state.rule_profile,
        'recent_boards': (state.recent_boards + (board,))[-GO2_RULES['recent_board_frames']:],
        'positional_history': state.positional_history | {board},
        'consecutive_passes': passes, 'actions': state.actions + (action,),
        'initial_board': state.initial_board, 'initial_to_play': state.initial_to_play,
        'initial_positional_history': state.initial_positional_history,
    }
    for name, value in fields.items(): object.__setattr__(child, name, value)
    return child


def go2_step(state, action):
    if state.terminal: raise Go2IllegalMove('Game ended after two consecutive passes.')
    if type(action) is not int or not 0 <= action <= state.pass_action:
        raise Go2IllegalMove('Action must be an integer board point or pass.')
    if action == state.pass_action:
        return go2_successor(state, state.board, state.consecutive_passes + 1, action)
    if state.board[action]: raise Go2IllegalMove('Point is occupied.')
    board = list(state.board); board[action] = state.to_play
    checked = set()
    for neighbor in go2_neighbors(state.size)[action]:
        if board[neighbor] == -state.to_play and neighbor not in checked:
            stones, liberties = go2_group(board, state.size, neighbor)
            checked.update(stones)
            if not liberties:
                for stone in stones: board[stone] = 0
    if not go2_group(board, state.size, action)[1]:
        raise Go2IllegalMove('Suicide is prohibited, including joined multi-group suicide.')
    values = tuple(board)
    if values in state.positional_history:
        raise Go2IllegalMove('Positional superko prohibits repetition of an earlier board.')
    return go2_successor(state, values, 0, action)


def go2_try_step(state, action):
    try: return go2_step(state, action)
    except Go2IllegalMove: return None


def go2_legal_actions(state):
    if state.terminal: return ()
    return tuple(action for action in range(state.pass_action)
                 if state.board[action] == 0 and go2_try_step(state, action) is not None) + (state.pass_action,)


def go2_legal_mask(state):
    legal = set(go2_legal_actions(state))
    return tuple(action in legal for action in range(state.pass_action + 1))


def go2_replay(initial_state, actions):
    state = initial_state
    for action in actions: state = go2_step(state, action)
    return state


def go2_verify_replay(state):
    root = go2_state_from_board(state.initial_board, to_play=state.initial_to_play,
                               size=state.size, komi=state.komi,
                               positional_history=state.initial_positional_history)
    reconstructed = go2_replay(root, state.actions)
    if reconstructed != state: raise ValueError('State history does not equal exact chronological replay.')
    return True


def go2_cache_key(state):
    """Exact immutable key for legal/search and network caches; no board-only aliasing."""
    return (state.rule_profile, state.size, state.komi, state.to_play,
            state.board, state.recent_boards, state.positional_history,
            state.consecutive_passes, state.actions, state.initial_board,
            state.initial_to_play, state.initial_positional_history)


def go2_state_digest(state):
    payload = json.dumps(go2_state_to_dict(state), sort_keys=True, separators=(',', ':'), allow_nan=False)
    return hashlib.sha256(payload.encode()).hexdigest()


## 3. Score and terminal signs

Ownership counts existing stones and empty regions bordered by only one color. Mixed-border and unbordered regions are neutral. White's score includes komi. Two passes make this frozen-rule outcome exact; before that point the score remains a static diagnostic.

In [ ]:
def go2_score(state):
    ownership = list(state.board)
    visited = set()
    for point, color in enumerate(state.board):
        if color or point in visited: continue
        region, pending, borders = {point}, [point], set()
        visited.add(point)
        while pending:
            for neighbor in go2_neighbors(state.size)[pending.pop()]:
                if state.board[neighbor]: borders.add(state.board[neighbor])
                elif neighbor not in visited:
                    visited.add(neighbor); region.add(neighbor); pending.append(neighbor)
        if len(borders) == 1:
            owner = next(iter(borders))
            for empty in region: ownership[empty] = owner
    black_area = ownership.count(1)
    white_area = ownership.count(-1)
    score_black = float(black_area - white_area - state.komi)
    return {
        'black_area': black_area, 'white_area': white_area,
        'neutral_points': ownership.count(0),
        'black_score': float(black_area), 'white_score': float(white_area + state.komi),
        'score_black': score_black, 'score_current': state.to_play * score_black,
        'ownership_black': tuple(ownership),
        'ownership_current': tuple(state.to_play * owner for owner in ownership),
        'terminal': state.terminal, 'rule_profile': state.rule_profile,
        'dead_stone_adjudication': False,
    }


def go2_outcome(state):
    scores = go2_score(state)
    winner = (1 if scores['score_black'] > 0 else -1 if scores['score_black'] < 0 else 0) if state.terminal else None
    return {**scores, 'winner': winner,
            'value_black': winner,
            'value_current': None if winner is None else state.to_play * winner,
            'status': 'TERMINAL' if state.terminal else 'UNKNOWN'}


## 4. Replay-safe D4 symmetry

Each of the eight square symmetries transforms every historical board, the setup root, the superko set, recent frames and every board-point action. Pass, side to play, komi and the rule profile remain invariant. Policy and ownership vectors use the same point mapping.

In [ ]:
def go2_transform_action(action, size, symmetry):
    if type(symmetry) is not int or not 0 <= symmetry < 8:
        raise ValueError('symmetry must be an integer from 0 to 7.')
    if type(action) is not int or not 0 <= action <= size * size:
        raise ValueError('Invalid action for D4 transformation.')
    if action == size * size: return action
    row, column = divmod(action, size)
    if symmetry >= 4: column = size - 1 - column
    for _ in range(symmetry % 4): row, column = column, size - 1 - row
    return row * size + column


def go2_transform_vector(values, size, symmetry):
    values = tuple(values)
    if len(values) not in (size * size, size * size + 1):
        raise ValueError('Vector must contain size² points and optionally pass.')
    transformed = [None] * len(values)
    for point in range(size * size):
        transformed[go2_transform_action(point, size, symmetry)] = values[point]
    if len(values) == size * size + 1: transformed[-1] = values[-1]
    return tuple(transformed)


def go2_transform_state(state, symmetry):
    transform = lambda board: go2_transform_vector(board, state.size, symmetry)
    return Go2State(
        board=transform(state.board), to_play=state.to_play, size=state.size,
        komi=state.komi, rule_profile=state.rule_profile,
        recent_boards=tuple(transform(board) for board in state.recent_boards),
        positional_history=frozenset(transform(board) for board in state.positional_history),
        consecutive_passes=state.consecutive_passes,
        actions=tuple(go2_transform_action(action, state.size, symmetry) for action in state.actions),
        initial_board=transform(state.initial_board), initial_to_play=state.initial_to_play,
        initial_positional_history=frozenset(transform(board) for board in state.initial_positional_history),
    )


## 5. Independent reference implementation

The reference uses a 2D board and recomputes every stone component by iterative label propagation. It does not call the fast engine's neighbors, group finder, transition or scorer. Its terminal and superko decisions follow the same declared contract. Agreement therefore checks two computation paths, with explicit fixtures supplying externally known outcomes.

In [ ]:
def go2_reference_components(matrix):
    """2D connected components via set expansion, independent of the fast DFS."""
    size = len(matrix)
    remaining = {(row, column) for row in range(size) for column in range(size) if matrix[row][column]}
    result = []
    while remaining:
        seed = min(remaining); color = matrix[seed[0]][seed[1]]
        component, frontier, liberties = {seed}, {seed}, set()
        while frontier:
            next_frontier = set()
            for row, column in frontier:
                for other in ((row-1,column),(row+1,column),(row,column-1),(row,column+1)):
                    rr, cc = other
                    if not (0 <= rr < size and 0 <= cc < size): continue
                    if matrix[rr][cc] == 0: liberties.add(other)
                    elif matrix[rr][cc] == color and other not in component:
                        component.add(other); next_frontier.add(other)
            frontier = next_frontier
        remaining.difference_update(component)
        result.append((color, component, liberties))
    return result


def go2_reference_transition(state, action):
    """Return an independent successor description, or None for an illegal action."""
    if state.consecutive_passes == 2 or type(action) is not int or not 0 <= action <= state.size ** 2:
        return None
    if action == state.size ** 2:
        return {'board': state.board, 'to_play': -state.to_play,
                'consecutive_passes': state.consecutive_passes + 1,
                'positional_history': state.positional_history}
    row, column = divmod(action, state.size)
    matrix = [list(state.board[start:start + state.size]) for start in range(0, len(state.board), state.size)]
    if matrix[row][column] != 0: return None
    matrix[row][column] = state.to_play
    # Enumerate all enemy components on the board; do not reuse fast local checks.
    for color, component, liberties in go2_reference_components(matrix):
        if color == -state.to_play and not liberties:
            for rr, cc in component: matrix[rr][cc] = 0
    for color, component, liberties in go2_reference_components(matrix):
        if (row, column) in component and not liberties: return None
    board = tuple(stone for line in matrix for stone in line)
    if board in state.positional_history: return None
    return {'board': board, 'to_play': -state.to_play, 'consecutive_passes': 0,
            'positional_history': state.positional_history | {board}}


def go2_reference_score(state):
    size = state.size
    matrix = [list(state.board[start:start + size]) for start in range(0, len(state.board), size)]
    ownership = dict(((row, column), matrix[row][column]) for row in range(size) for column in range(size))
    empties = {point for point, color in ownership.items() if color == 0}
    while empties:
        seed = min(empties); region = {seed}; changed = True
        while changed:
            changed = False
            for point in tuple(empties - region):
                row, column = point
                if any(other in region for other in ((row-1,column),(row+1,column),(row,column-1),(row,column+1))):
                    region.add(point); changed = True
        borders = set()
        for row, column in region:
            for rr, cc in ((row-1,column),(row+1,column),(row,column-1),(row,column+1)):
                if 0 <= rr < size and 0 <= cc < size and matrix[rr][cc]: borders.add(matrix[rr][cc])
        owner = next(iter(borders)) if len(borders) == 1 else 0
        for point in region: ownership[point] = owner
        empties.difference_update(region)
    values = tuple(ownership[(row,column)] for row in range(size) for column in range(size))
    return {'ownership_black': values, 'score_black': float(sum(values) - state.komi)}


def go2_independent_check(state, *, all_actions=True):
    """Check exact legal mask/transitions and area ownership without trusting fast witnesses."""
    actions = range(state.pass_action + 1) if all_actions else ()
    for action in actions:
        fast = go2_try_step(state, action)
        reference = go2_reference_transition(state, action)
        if (fast is None) != (reference is None):
            raise AssertionError(f'Legal-action disagreement at action {action}.')
        if fast is not None:
            for key in ('board','to_play','consecutive_passes','positional_history'):
                if getattr(fast, key) != reference[key]:
                    raise AssertionError(f'Independent successor disagreement: {key}.')
    scores, reference = go2_score(state), go2_reference_score(state)
    if scores['ownership_black'] != reference['ownership_black'] or scores['score_black'] != reference['score_black']:
        raise AssertionError('Independent area/ownership disagreement.')
    return True


## 6. Deterministic engine fixtures

The checks include known captures, suicide joining existing groups, positional ko, pass termination, exact area ownership, terminal perspective signs, history-sensitive cache separation, serialization/replay and D4 covariance. Random trajectories add independent whole-game checks; they are engineering fixtures, not training examples or strength evidence.

In [ ]:
import random
import time
from pathlib import Path
from IPython.display import HTML, display

validation_started = time.monotonic()
validation_checks = []

def record_check(name, check):
    check()
    validation_checks.append({'check': name, 'status': 'PASS'})


def fixture_capture():
    # White at (1,1) has exactly one liberty (1,2).
    root = go2_state_from_board((0,1,0,1,-1,0,0,1,0), size=3, komi=0.5)
    child = go2_step(root, 5)
    assert child.board[4] == 0 and child.board[5] == 1
    assert child.actions == (5,) and child.to_play == -1
    go2_independent_check(root); go2_independent_check(child)
    # Four distinct enemy groups are captured before suicide is considered.
    values = [0] * 25
    for point in (7,11,13,17): values[point] = -1
    for point in (2,6,8,10,14,16,18,22): values[point] = 1
    surrounded = go2_state_from_board(values, size=5)
    escape_capture = go2_step(surrounded, 12)
    assert all(escape_capture.board[point] == 0 for point in (7,11,13,17))
    assert escape_capture.board[12] == 1 and go2_group(escape_capture.board,5,12)[1] == frozenset((7,11,13,17))
    go2_independent_check(surrounded); go2_independent_check(escape_capture)


def fixture_suicide():
    root = go2_state_from_board((0,-1,0,-1,0,-1,0,-1,0), size=3)
    assert go2_try_step(root, 4) is None
    # Two separate friendly groups share their last liberty at center;
    # joining them creates a multi-stone suicide, rather than an isolated stone.
    values = [0] * 25
    for point in (7,17): values[point] = 1
    for point in (2,6,8,11,13,16,18,22): values[point] = -1
    joined = go2_state_from_board(values, size=5)
    assert go2_try_step(joined, 12) is None
    go2_independent_check(root); go2_independent_check(joined)


def fixture_ko_and_history():
    # Use an explicit conventional 5x5 ko with interior capture/recapture.
    values = [0] * 25
    for point in (7,11,17): values[point] = 1
    for point in (12,8,14,18): values[point] = -1
    root = go2_state_from_board(values, size=5)
    captured = go2_step(root, 13)
    assert captured.board[12] == 0 and captured.board[13] == 1
    assert go2_try_step(captured, 12) is None
    without_old = go2_state_from_board(captured.board, to_play=-1, size=5)
    assert go2_try_step(without_old, 12) is not None
    # The same player, board, pass count and frames have different legal moves.
    assert root.board in captured.positional_history
    restricted = go2_state_from_board(captured.board, to_play=-1, size=5,
                                     positional_history=(root.board,))
    assert go2_cache_key(without_old) != go2_cache_key(restricted)
    assert go2_legal_mask(without_old)[12] and not go2_legal_mask(restricted)[12]
    # A distant historical repeat must also be rejected, not only immediate ko.
    legal = go2_step(without_old, 12)
    distant = go2_state_from_board(without_old.board, to_play=-1, size=5,
                                  positional_history=(legal.board, (0,)*25))
    assert go2_try_step(distant, 12) is None
    go2_independent_check(root); go2_independent_check(captured)


def fixture_pass_and_signs():
    root = go2_initial_state(size=3, komi=0.5)
    first = go2_step(root, root.pass_action)
    terminal = go2_step(first, first.pass_action)
    assert terminal.terminal and go2_legal_actions(terminal) == ()
    assert go2_legal_mask(terminal) == (False,) * 10
    assert terminal.positional_history == root.positional_history
    assert go2_try_step(terminal, 0) is None
    outcome = go2_outcome(terminal)
    assert outcome['winner'] == -1 and outcome['value_black'] == -1 and outcome['value_current'] == -1
    white_root = go2_state_from_board(root.board, size=3, to_play=-1, komi=0.5)
    white_terminal = go2_replay(white_root, (9,9))
    assert go2_outcome(white_terminal)['value_current'] == 1
    black_setup = go2_state_from_board((1,0,0,0,0,0,0,0,0), size=3, komi=0.5)
    black_terminal = go2_replay(black_setup,(9,9))
    assert go2_outcome(black_terminal)['winner'] == 1 and go2_outcome(black_terminal)['value_current'] == 1
    assert go2_outcome(root)['status'] == 'UNKNOWN' and go2_outcome(root)['value_current'] is None
    main = go2_initial_state()
    white_to_pass = go2_step(main, 81)
    finished = go2_step(white_to_pass, 81)
    assert go2_outcome(finished)['winner'] == -1 and go2_outcome(finished)['value_current'] == -1
    assert -go2_outcome(finished)['value_current'] == 1 and white_to_pass.to_play == -1
    for state in (root,first,terminal,white_terminal): go2_independent_check(state)


def fixture_area():
    # A single black stone borders the only empty region: all nine points count Black.
    one = go2_state_from_board((1,0,0,0,0,0,0,0,0), size=3, komi=0.5)
    score = go2_score(one)
    assert score['black_area'] == 9 and score['white_area'] == 0 and score['score_black'] == 8.5
    # Opposite colors border the same region, making all seven empty points neutral.
    mixed = go2_state_from_board((1,0,0,0,0,0,0,0,-1), size=3, komi=0.5)
    assert go2_score(mixed)['neutral_points'] == 7 and go2_score(mixed)['score_black'] == -0.5
    # White center is still counted White even if a reader would call it dead.
    values = [1] * 25; values[12] = -1; values[13] = 0
    enclosed = go2_state_from_board(values, size=5, komi=0.5)
    score = go2_score(enclosed)
    assert score['white_area'] == 1 and score['black_area'] == 23 and score['neutral_points'] == 1 and score['score_black'] == 21.5
    for state in (one,mixed,enclosed):
        assert go2_score(state)['black_area'] + go2_score(state)['white_area'] + go2_score(state)['neutral_points'] == state.size ** 2
        go2_independent_check(state, all_actions=False)


def fixture_replay_and_symmetry():
    state = go2_replay(go2_initial_state(), (40,31,30,41,49,32,81,48))
    restored = go2_state_from_dict(json.loads(json.dumps(go2_state_to_dict(state))))
    assert restored == state and go2_verify_replay(restored)
    assert len(state.recent_boards) == 8 and len(state.actions) == 8
    assert go2_state_digest(restored) == go2_state_digest(state)
    old = go2_state_to_dict(state); old['to_play'] = -state.to_play
    try: go2_state_from_dict(old)
    except ValueError: pass
    else: raise AssertionError('Chronology/player corruption accepted.')
    scores = go2_score(state)
    legal = go2_legal_mask(state)
    for symmetry in range(8):
        transformed = go2_transform_state(state, symmetry)
        assert go2_verify_replay(transformed)
        assert go2_legal_mask(transformed) == go2_transform_vector(legal, state.size, symmetry)
        assert go2_score(transformed)['score_black'] == scores['score_black']
        assert go2_score(transformed)['ownership_black'] == go2_transform_vector(scores['ownership_black'], state.size, symmetry)
        for action in go2_legal_actions(state):
            lhs = go2_transform_state(go2_step(state, action), symmetry)
            rhs = go2_step(transformed, go2_transform_action(action,state.size,symmetry))
            assert lhs == rhs
    # Chronological recent frames affect a network cache even when board/player agree.
    empty = go2_initial_state()
    passed = go2_replay(empty, (81,81))
    assert empty.board == passed.board and empty.to_play == passed.to_play
    assert go2_cache_key(empty) != go2_cache_key(passed)


for label, fixture in (
    ('Single/multiple-group capture before suicide; independent transitions',fixture_capture),
    ('Single-stone and joined multi-group suicide rejection',fixture_suicide),
    ('Immediate and distant positional superko; history-sensitive cache',fixture_ko_and_history),
    ('Pass exemption, two-pass terminal, no terminal actions and perspective signs',fixture_pass_and_signs),
    ('Area ownership, neutral territory and no automatic dead-stone removal',fixture_area),
    ('Complete JSON replay, chronology corruption rejection and D4 covariance',fixture_replay_and_symmetry),
): record_check(label, fixture)


In [ ]:
random_game_rows = []
for game_id in range(6):
    rng = random.Random(2026100600 + game_id)
    state = go2_initial_state()
    placements = 0
    # Each complete 9x9 engineering game ends by two actual legal passes.
    while not state.terminal:
        go2_independent_check(state)
        legal = go2_legal_actions(state)
        nonpass = [action for action in legal if action != state.pass_action]
        if placements >= 96 or not nonpass or (placements >= 30 and rng.random() < 0.06):
            action = state.pass_action
        else:
            action = rng.choice(nonpass); placements += 1
        state = go2_step(state, action)
    go2_independent_check(state)
    assert go2_verify_replay(state)
    assert go2_state_from_dict(go2_state_to_dict(state)) == state
    result = go2_outcome(state)
    random_game_rows.append({'game':game_id,'plies':state.move_number,
                             'terminal':'two legal passes','Black area margin':result['score_black'],
                             'winner':'Black' if result['winner']==1 else 'White' if result['winner']==-1 else 'Draw'})
validation_checks.append({'check':'Six complete random 9×9 games: every legal action, successor and ownership independently checked', 'status':'PASS'})
validation_seconds = time.monotonic() - validation_started
print(f"PASS — {len(validation_checks)} check groups; {sum(row['plies'] for row in random_game_rows)} complete-game plies; {validation_seconds:.2f} seconds on CPU.")


## 7. Saved validation results

These results validate the declared rule engine and replay contract. They do not establish playing strength, a learning benefit or full-game search optimality. Small-board setup fixtures are deliberately separated from the six empty-board 9×9 trajectories.

In [ ]:
def go2_html_table(rows):
    import html
    columns = tuple(rows[0]) if rows else ()
    header = ''.join(f'<th>{html.escape(str(column))}</th>' for column in columns)
    body = ''.join('<tr>' + ''.join(f'<td>{html.escape(str(row[column]))}</td>' for column in columns) + '</tr>' for row in rows)
    return '<table style="border-collapse:collapse;width:100%;font-size:14px"><thead><tr>'+header+'</tr></thead><tbody>'+body+'</tbody></table>'

report_dir = Path.cwd() / 'reports' / 'ahr_v2'
report_dir.mkdir(parents=True, exist_ok=True)
notebook_path = Path.cwd() / 'experiments' / 'ahr_v2' / '01_rules_and_replay.ipynb'
# Load only shared tagged definitions for the canonical project source identity.
# This does not execute shared recovery fixtures or any upstream workload.
import nbformat
receipt_namespace = {'__name__': 'go2_rules_receipt_identity'}
shared_path = Path.cwd() / 'experiments' / 'ahr_v2' / '00_contracts_and_recovery.ipynb'
for shared_cell in nbformat.read(shared_path,4).cells:
    if shared_cell.cell_type == 'code' and 'ahr-export' in shared_cell.metadata.get('tags',[]):
        exec(compile(shared_cell.source,str(shared_path)+':ahr-export','exec'),receipt_namespace)
export_sha = receipt_namespace['go2_code_sha'](notebook_path)
source_identity = receipt_namespace['go2_source_identity'](Path.cwd(),require_all=False)
receipt = {'schema':'go2_rules_validation_v1','status':'PASS','definition_sha256':export_sha,
           'definition_sha256_algorithm':'canonical JSON list of tagged source strings',
           'sources':source_identity,
           'rules':GO2_RULES,'checks':validation_checks,'random_complete_games':random_game_rows,
           'complete_game_plies':sum(row['plies'] for row in random_game_rows),
           'runtime_seconds':round(validation_seconds,3),'checkpoint':'not applicable: deterministic CPU fixtures',
           'source_models_imported':False,'source_training_imported':False}
receipt_path = report_dir / 'rules_validation.json'
temporary = receipt_path.with_suffix('.json.tmp')
with temporary.open('w') as stream:
    json.dump(receipt,stream,indent=2,allow_nan=False); stream.flush()
    import os
    os.fsync(stream.fileno())
os.replace(temporary,receipt_path)
display(HTML('<h3>Rule and replay checks</h3>'+go2_html_table(validation_checks)))
display(HTML('<h3>Independent complete-game fixtures</h3>'+go2_html_table(random_game_rows)))
print(f"Validation receipt: {receipt_path.relative_to(Path.cwd())}")
print(f"Tagged definition SHA-256: {export_sha}")


## Exact replay and downstream use

Only execute cells tagged `ahr-export` when importing this notebook. Definitions have no dependency on old runs. For a saved state, restore with `go2_state_from_dict`, then require `go2_verify_replay` at ingestion; malformed or inconsistent histories must raise rather than be silently repaired. Apply `go2_transform_state` to augment a trajectory, and `go2_transform_vector` to its policy and ownership labels. Store the rule profile and tagged definition SHA-256 with every new dataset, run and checkpoint.

Move/time limits belong to the caller: reaching a cutoff is `UNKNOWN` and cannot be converted into an exact terminal outcome. This engine has no automatic cutoff, dead-stone heuristic or resignation rule.